# ML-Based Parsing

**Module 2 — Lesson 8**

Rule-based parsers encode what you know — every template is a layout you've explicitly studied and encoded. ML-based approaches take a different path: they learn from examples, which means they can generalize to layouts and noise patterns you haven't written rules for.

This notebook walks through three approaches to ML-based email header extraction:

1. **Default NER** — what a general-purpose model finds in email headers, and where it falls short
2. **Finetuned NER** — training a model on email-specific labels, from 20 examples to 1,100+
3. **GLiNER2** — zero-shot extraction with no training data at all


## 1. Named entity recognition

Named entity recognition (NER) is the task of finding and classifying spans of text — identifying that "Kay Mann" is a person, "Thursday, May 31" is a date, or "enron.com" is part of an email address.

spaCy ships with models trained on news and web text that can identify general entity types like PERSON, DATE, ORG, and GPE (geopolitical entity). These models have never seen email headers, so they don't know anything about senders, recipients, or subject lines — but they do understand English text well enough to find names and dates.

Run the cells below to load spaCy's smallest English model and see what it finds in our email headers.


In [1]:
!pip install spacy -q
!python -m spacy download en_core_web_sm -q


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [6]:
import spacy
import re
from pathlib import Path

nlp = spacy.load("en_core_web_sm")

TXT_DIR = Path("data/extracted_text")
txt_files = sorted(TXT_DIR.glob("*.txt"))

BOILERPLATE = [
    r"^'?CONFIDENTIAL.*$", r"^'?UNCLASSIFIED.*$", r"^Enron\s*Corp.*$",
    r"^Case\s*No[,\.].*$", r"^Doc\s*No[,\.].*$", r"^Date:?\s*\d",
    r"^ENRON\s*CORP.*$", r"^SUBJECT TO PROTECTIVE.*$", r"^RELEASE IN.*$",
    r"^ENRON[-=]\d+.*$",
]

def strip_boilerplate(text):
    lines = text.split("\n")
    cleaned = [l for l in lines if not any(re.match(p, l.strip(), re.IGNORECASE) for p in BOILERPLATE)]
    return "\n".join(cleaned).strip()

print(f"Loaded {len(txt_files):,} files, spaCy model: {nlp.meta['name']}")


Loaded 4,911 files, spaCy model: core_web_sm


In [7]:
# Run default NER on a sample email
sample = txt_files[0].read_text(encoding="utf-8")
cleaned = strip_boilerplate(sample)
header_text = "\n".join(cleaned.split("\n")[:15])

doc = nlp(header_text)

print("Email header:")
print(header_text)
print()
print("Default NER finds:")
for ent in doc.ents:
    print(f"  [{ent.label_:10s}] {ent.text!r}")


Email header:
From:
Rob Bradley <rob.bradley@enron.com>
Sent:
Tue, 21 Nov 2000 08:34:00 -0800 (PST)
To:
Kenneth Lay <kenneth.lay@enron.com>
Cc:
Alhamd Alkhayat <alhamd.alkhayat@enron.com>
Subject:
Remarks to EES Employees--December 1
[REDACTED] B6
EES employee meeting next Friday, and I put together the attached as a
potential framework.  It provides an Enron problem list of 1985/86 versus
today and a brief look at our different visions.
If I can do something else, I'll be back Monday morning and will be happy to

Default NER finds:
  [PERSON    ] 'Rob Bradley'
  [DATE      ] '21 Nov 2000'
  [ORG       ] 'PST'
  [PERSON    ] 'Kenneth Lay'
  [PERSON    ] 'Alhamd Alkhayat'
  [ORG       ] 'EES Employees'
  [DATE      ] 'December 1'
  [ORG       ] 'EES'
  [DATE      ] 'next Friday'
  [ORG       ] 'Enron'
  [CARDINAL  ] '1985/86'
  [DATE      ] 'today'
  [DATE      ] 'Monday'
  [TIME      ] 'morning'


The model identifies names and dates — useful in general, but it has no way to tell a sender from a recipient. Every name is just PERSON. There's no concept of a subject line or an email address.

Run the next cell to see how consistent this is across five emails.


In [8]:
print("Default NER across 5 emails:\n")
for txt_path in txt_files[:5]:
    text = txt_path.read_text(encoding="utf-8")
    cleaned = strip_boilerplate(text)
    header_text = "\n".join(cleaned.split("\n")[:15])
    doc = nlp(header_text)
    
    print(f"=== {txt_path.name} ===")
    for ent in doc.ents:
        print(f"  {ent.label_:10s} -> {ent.text!r}")
    if not doc.ents:
        print("  (no entities found)")
    print()


Default NER across 5 emails:

=== E0000813E.txt ===
  PERSON     -> 'Rob Bradley'
  DATE       -> '21 Nov 2000'
  ORG        -> 'PST'
  PERSON     -> 'Kenneth Lay'
  PERSON     -> 'Alhamd Alkhayat'
  ORG        -> 'EES Employees'
  DATE       -> 'December 1'
  ORG        -> 'EES'
  DATE       -> 'next Friday'
  ORG        -> 'Enron'
  CARDINAL   -> '1985/86'
  DATE       -> 'today'
  DATE       -> 'Monday'
  TIME       -> 'morning'

=== E0000D1D0.txt ===
  ORG        -> 'EDIS Email Service'
  PERSON     -> 'Fri'
  CARDINAL   -> '5'
  DATE       -> '2002'
  ORG        -> 'PST'
  PERSON     -> 'Motley'
  PERSON     -> 'Matt'
  CARDINAL   -> '2'
  ORG        -> 'Southern California Seismic Network'
  ORG        -> 'Caltech'
  ORG        -> 'USGS'
  CARDINAL   -> '1'
  QUANTITY   -> '4.2  Ml\nTime        :'
  DATE       -> '2002'
  ORG        -> '00:02:55 AM PST'

=== E000B745B.txt ===
  PERSON     -> 'Susan Scott'
  DATE       -> '31 May 2000'
  TIME       -> '02:24:00'
  PERSON     -> 'D

The model reliably finds names and dates, but it can't distinguish between the sender and the recipients — both are PERSON.

We could use these results in tandem with the patterns from the previous lesson -- but there's another, simpler way.

For our graph, we'd ideally like labels that map to specific fields: SENDER, RECIPIENT, CC_RECIPIENT, DATE, SUBJECT, and email addresses.

To teach a model these email-specific labels, we need to **finetune** it — take a model that already understands English, and train it further on labelled email examples so it learns the new task.


## 2. Finetuned NER (20 examples)

The model below has been finetuned on just 20 annotated email headers. Instead of generic entity types, it uses labels designed for email parsing:

- **SENDER** / **SENDER_EMAIL** — the sender's name and email address
- **RECIPIENT** / **RECIPIENT_EMAIL** — each To recipient's name and email
- **CC_RECIPIENT** / **CC_RECIPIENT_EMAIL** — each Cc recipient's name and email
- **DATE** — the sent date
- **SUBJECT** — the subject line

With only 20 examples, the model has seen the task but hasn't seen enough variety to learn it properly. Load it and see what it gets right — and what it gets wrong.

In [9]:
nlp_20 = spacy.load("data/ner_models/ner_20/model-best")
print(f"Labels: {nlp_20.get_pipe('ner').labels}")


Labels: ('CC_RECIPIENT', 'CC_RECIPIENT_EMAIL', 'DATE', 'RECIPIENT', 'RECIPIENT_EMAIL', 'SENDER', 'SENDER_EMAIL', 'SUBJECT')


In [ ]:
print("Finetuned NER (20 examples) across 5 emails:\n")
for txt_path in txt_files[:5]:
    text = txt_path.read_text(encoding="utf-8")
    cleaned = strip_boilerplate(text)
    header_text = "\n".join(cleaned.split("\n")[:15])
    doc = nlp_20(header_text)

    print(f"=== {txt_path.name} ===")
    if doc.ents:
        for ent in doc.ents:
            print(f"  {ent.label_:15s} -> {ent.text!r}")
    else:
        print("  (no entities found)")
    print()

The model finds entities in roughly the right places — it knows that names and email addresses matter. But look at the *labels*: `Rob Bradley` is tagged SUBJECT, not SENDER. Every email address is SENDER_EMAIL, whether it belongs to the sender or a recipient. The sender in E00292463 is labelled RECIPIENT.

20 examples is enough to teach the model *where* entities are, but not *which label* to assign. It confuses senders with recipients, and subjects with names, because it hasn't seen enough examples to learn what distinguishes them.

## 3. Training data

NER training data pairs each text with a set of annotations — character offsets marking where each entity starts and ends, and which label it belongs to.

The annotations for this course are stored as JSONL (one JSON object per line). Each line contains the email header text and its entity spans. Run the next cell to see what one annotated example looks like.


In [11]:
import json as _json

TRAIN_DATA_PATH = Path("data/ner_training/annotations.jsonl")

with open(TRAIN_DATA_PATH) as f:
    raw_line = f.readline()

print("Raw JSONL line:")
print(raw_line[:1200])
print("...")
print()

example = _json.loads(raw_line)
print(f"Keys: {list(example.keys())}")
print(f"Text length: {len(example['text'])} chars")
print(f"Spans: {len(example['spans'])}")
print()
for span in example["spans"]:
    s, e, label = span["start"], span["end"], span["label"]
    print(f"  {label:22s}  ->  {example['text'][s:e]!r}")


Raw JSONL line:
{"text": "From:\nJeffrey Porter <jeffrey.porter@enron.com>\nSent:\nMon, 20 Nov 2000 00:56:00 -0800 (PST)\nTo:\njoann.collins@enron.com, Alvin Thompson <alvin.thompson@enron.com>,\nChris Germany <chris.germany@enron.com>\nCc:\nMichael H Garred <michael.garred@enron.com>\nSubject:\nNOV Rest of Month\nWe would like to use 5,000 dth/d  of CES' TCO K#68918 to deliver to COH 5-2\nfor ROM. Please pull from the pool as TCO said we should be OK. Any problem?", "spans": [{"start": 6, "end": 20, "label": "SENDER"}, {"start": 22, "end": 46, "label": "SENDER_EMAIL"}, {"start": 54, "end": 91, "label": "DATE"}, {"start": 96, "end": 119, "label": "RECIPIENT_EMAIL"}, {"start": 121, "end": 135, "label": "RECIPIENT"}, {"start": 137, "end": 161, "label": "RECIPIENT_EMAIL"}, {"start": 164, "end": 177, "label": "RECIPIENT"}, {"start": 179, "end": 202, "label": "RECIPIENT_EMAIL"}, {"start": 208, "end": 224, "label": "CC_RECIPIENT"}, {"start": 226, "end": 250, "label": "CC_RECIPIENT_EMAIL"}, {

The training data contains ~1,000 annotated email headers. 150 were hand-labelled in Label Studio, and the remainder were annotated by an LLM using the hand-labelled examples as few-shot context. The LLM annotations were not individually reviewed -- they're silver-standard, not gold. Some will have wrong span boundaries or mislabelled entities.

spaCy models are remarkably tolerant to this kind of noise. The volume of data compensates for individual errors, and the model learns the dominant pattern rather than memorising noise. If you're building your own pipeline, spot-check a sample of LLM annotations before training -- but don't expect to review all of them.

### Annotation tools

The annotations for this course were created in [Label Studio](https://labelstud.io/), an open-source annotation tool that you can self-host for free. [Prodigy](https://prodi.gy/), built by the spaCy team, is a paid alternative that integrates directly with spaCy — annotation, format conversion, and training all happen within a single workflow.

Each tool has its own export format, and converting between annotation formats and spaCy's binary `.spacy` training format is a common friction point. For this course, the `.spacy` files are pre-built so you can focus on training.


In [12]:
from spacy.tokens import DocBin

train_db = DocBin().from_disk("data/ner_training/train.spacy")
dev_db = DocBin().from_disk("data/ner_training/dev.spacy")
print(f"Train: {len(train_db)} documents")
print(f"Dev:   {len(dev_db)} documents")


Train: 895 documents
Dev:   15 documents


## 4. Training

spaCy training requires two things: a config file (which defines the model architecture and training parameters) and the `.spacy` data files.

The config below uses spaCy's `efficiency` preset — a lightweight tok2vec encoder that runs on CPU. The model trains from blank, meaning there are no pretrained word vectors. Everything the model learns comes from the annotations.


In [13]:
!python -m spacy init config data/ner_training/config_blank.cfg \
    --pipeline ner --lang en --optimize efficiency -F


ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
data/ner_training/config_blank.cfg
You can now add your data and train your pipeline:
python -m spacy train config_blank.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [14]:
!python -m spacy train data/ner_training/config_blank.cfg \
    --paths.train data/ner_training/train.spacy \
    --paths.dev data/ner_training/dev.spacy \
    --output data/ner_models/ner_blank


ℹ Saving to output directory: data/ner_models/ner_blank
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     35.06    0.00    0.00    0.00    0.00
  0     200       2163.66   4440.55   55.63   58.52   53.02    0.56
  0     400        255.56   1993.12   80.28   82.86   77.85    0.80
  0     600        235.87    910.75   86.39   87.59   85.23    0.86
  1     800        596.02   1439.79   88.74   90.28   87.25    0.89
  1    1000        264.51   1125.25   88.66   90.85   86.58    0.89
  2    1200        602.32   1482.12   86.71   90.51   83.22    0.87
  2    1400        781.44   177

### Reading the training output

The training log shows a table of metrics at each evaluation step. The columns that matter most:

- **ENTS_P** (precision) — of the entities the model found, what percentage were correct?
- **ENTS_R** (recall) — of the entities that actually exist, what percentage did the model find?
- **ENTS_F** (F1 score) — the harmonic mean of precision and recall. This single number balances both: a model that finds everything but gets half of them wrong scores poorly, and so does a model that's always right but misses most entities.

An F1 of 89 means the model is finding most entities and getting most of them right. An F1 of 53 (what the 20-example model achieved) means it finds spans but can't reliably assign the correct label.

spaCy saves the best model (highest F1 on the dev set) automatically.

## 5. Trained NER (1,000+ examples)

Load the model you just trained and compare it to the undertrained model and the default NER — all three on the same five emails, side by side.


In [15]:
nlp_trained = spacy.load("data/ner_models/ner_blank/model-best")
print(f"Labels: {nlp_trained.get_pipe('ner').labels}")


Labels: ('CC_RECIPIENT', 'CC_RECIPIENT_EMAIL', 'DATE', 'RECIPIENT', 'RECIPIENT_EMAIL', 'SENDER', 'SENDER_EMAIL', 'SUBJECT')


In [16]:
print("Side-by-side comparison on 5 emails:\n")
for txt_path in txt_files[:5]:
    text = txt_path.read_text(encoding="utf-8")
    cleaned = strip_boilerplate(text)
    header_text = "\n".join(cleaned.split("\n")[:15])
    
    doc_default = nlp(header_text)
    doc_20 = nlp_20(header_text)
    doc_full = nlp_trained(header_text)
    
    print(f"=== {txt_path.name} ===")
    print(f"  Default NER:")
    for ent in doc_default.ents:
        print(f"    {ent.label_:15s} -> {ent.text!r}")
    if not doc_default.ents:
        print(f"    (none)")
    print(f"  Undertrained (20):")
    for ent in doc_20.ents:
        print(f"    {ent.label_:15s} -> {ent.text!r}")
    if not doc_20.ents:
        print(f"    (none)")
    print(f"  Trained (1,100+):")
    for ent in doc_full.ents:
        print(f"    {ent.label_:15s} -> {ent.text!r}")
    if not doc_full.ents:
        print(f"    (none)")
    print()


Side-by-side comparison on 5 emails:

=== E0000813E.txt ===
  Default NER:
    PERSON          -> 'Rob Bradley'
    DATE            -> '21 Nov 2000'
    ORG             -> 'PST'
    PERSON          -> 'Kenneth Lay'
    PERSON          -> 'Alhamd Alkhayat'
    ORG             -> 'EES Employees'
    DATE            -> 'December 1'
    ORG             -> 'EES'
    DATE            -> 'next Friday'
    ORG             -> 'Enron'
    CARDINAL        -> '1985/86'
    DATE            -> 'today'
    DATE            -> 'Monday'
    TIME            -> 'morning'
  Undertrained (50):
    SUBJECT         -> 'Rob Bradley'
    SENDER_EMAIL    -> 'rob.bradley@enron.com'
    DATE            -> 'Tue, 21 Nov 2000 08:34:00 -0800 (PST)'
    SUBJECT         -> 'Kenneth Lay <'
    SENDER_EMAIL    -> 'kenneth.lay@enron.com'
    SUBJECT         -> 'Alhamd Alkhayat'
    SENDER_EMAIL    -> 'alhamd.alkhayat@enron.com'
    SUBJECT         -> 'Remarks to EES Employees--December 1'
    RECIPIENT       -> '[REDACTED] 

The difference is visible immediately. The default model finds generic PERSON and DATE entities. The undertrained model finds spans in the right places but assigns the wrong labels. The trained model reliably distinguishes senders from recipients, finds subjects, and handles multi-recipient lists.

## 6. Model comparison

The model you just trained uses spaCy's smallest architecture — a lightweight tok2vec encoder that runs on CPU. 

spaCy also offers transformer-based models. We benchmarked a transformer (RoBERTa) on the same training data to see how much the architecture matters.

The tok2vec model was trained from blank (no pretrained word vectors). The transformer uses a pretrained RoBERTa encoder — the language model's existing knowledge of English gives it a significant head start.

| Model | F1 | Notes |
|---|---|---|
| **tok2vec (blank, 20 examples)** | 52.6 | Finds spans, wrong labels |
| **tok2vec (blank, 1,000 examples)** | 89.4 | Reliable on most labels |
| **transformer (RoBERTa, 1,000 examples)** | **96.3** | Near-perfect, requires GPU |

The tok2vec model at 89.4 is strong for a CPU model. The transformer reaches 96.3 — the gap is largest on context-dependent labels like CC_RECIPIENT and email addresses, where a pretrained language model excels.

The Colab training notebook is at `data/ner_training/colab_trf_benchmark.ipynb`.

To learn more about using spaCy in general, see the [spaCy docs](https://spacy.io/).

## 7. GLiNER2: zero-shot extraction

GLiNER2 takes a completely different approach. Instead of training on labelled examples, you describe the entities you want in natural language — "sender name", "recipient email address", "email subject" — and the model finds matching spans at inference time. No annotation, no training, no config files.

The trade-off is that you have less control over exactly what the model extracts, and you can't improve it by adding more examples. But for rapid prototyping, or when you don't have the budget for annotation, it's a strong starting point.


In [13]:
!pip install gliner2 -q


In [17]:
from gliner2 import GLiNER2

print("Loading GLiNER2 model (fastino/gliner2-base-v1)...")
gliner_model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
print("Loaded.")


Loading GLiNER2 model (fastino/gliner2-base-v1)...
🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Loaded.


In [18]:
labels = [
    "sender name", "sender email address",
    "recipient name", "recipient email address",
    "sent date", "email subject",
]

print("GLiNER2 results across 5 emails:\n")
for txt_path in txt_files[:5]:
    text = txt_path.read_text(encoding="utf-8")
    cleaned = strip_boilerplate(text)
    header_text = "\n".join(cleaned.split("\n")[:20])
    
    result = gliner_model.extract_entities(header_text, labels)
    
    print(f"=== {txt_path.name} ===")
    for label, values in result.get("entities", {}).items():
        if values:
            for v in values:
                print(f"  {label:30s} -> {v!r}")
    print()


GLiNER2 results across 5 emails:

=== E0000813E.txt ===
  sender name                    -> 'Rob Bradley'
  sender email address           -> 'rob.bradley@enron.com'
  recipient name                 -> 'Kenneth Lay'
  recipient email address        -> 'kenneth.lay@enron.com'
  email subject                  -> 'Remarks to EES Employees--December 1'

=== E0000D1D0.txt ===
  sender name                    -> 'EDIS Email Service'
  sender email address           -> 'edismail@incident.com'
  recipient name                 -> 'Motley, Matt'
  recipient email address        -> 'matt.motley@enron.com'

=== E000B745B.txt ===
  sender name                    -> 'Susan Scott'
  sender name                    -> 'Christine Stokes'
  sender email address           -> 'susan.scott@enron.com'
  recipient name                 -> 'Lorraine Lindberg'
  recipient name                 -> 'Glen\nHass'
  recipient name                 -> 'Bill\nCordes'
  recipient name                 -> 'Kevin Hyatt'
  re

GLiNER2 does a strong job here — it correctly distinguishes sender from recipient, finds subjects reliably, and handles multi-recipient lists with no training data at all. The main errors come from long multi-line recipient lists and OCR noise rather than model limitations.

## 8. Assessment

| | Default NER | Finetuned (20) | Finetuned (1,000+) | Transformer (1,000+) | GLiNER2 (zero-shot) |
|---|---|---|---|---|---|
| Training data | None | 20 examples | 1,000+ examples | 1,000+ examples | None |
| Overall F1 | — | 52.6 | 89.4 | 96.3 | N/A (no shared eval format) |
| Sender/recipient | No | Partial (wrong labels) | Yes | Yes | Yes |
| CC handling | No | No | Yes | Yes | Partial |
| Speed | Fast | Fast | Fast | Slow (GPU) | Medium |
| Setup cost | None | 20 labels (~15 min) | 150 labels + LLM batch | 150 labels + LLM batch + GPU | None |

### Other approaches

**spaCy spancat** classifies overlapping zones in a document. Where NER finds individual entities (a name, a date), spancat can identify the entire `To:` block as a zone, then NER can extract individual recipients within it. This two-stage approach — zone detection followed by entity extraction — is effective when headers have complex or variable layouts. It requires annotated training data in the same way NER does.

**Prodigy** is an annotation tool from the spaCy team that integrates directly with spaCy's training pipeline. It provides an efficient interface for creating NER and spancat training data, with active learning to prioritise the most informative examples. If you're building models for your own corpus, Prodigy significantly reduces the time from annotation to trained model.

**Label Studio** is an open-source alternative for data annotation. It supports NER, span labeling, and many other annotation types. Free to self-host, and a good option when you need flexibility without a licence cost.


## Summary

- **Named entity recognition** finds and classifies spans of text. Default spaCy models identify general entities (PERSON, DATE) but can't distinguish email-specific fields like sender vs recipient.
- **Finetuning** teaches a model new labels from annotated examples. 20 examples teach the model where entities are; 1,000+ teach it which labels to assign (89.4 F1 on CPU, 96.3 with a transformer on GPU).
- **Training from blank** outperforms pretrained word vectors on this structural task — the annotations alone teach the encoder everything it needs.
- **GLiNER2** achieves strong results with no training data — describe the entities you want in natural language and the model finds them.
- **Annotation tools** like Label Studio (free) and Prodigy (paid, spaCy-integrated) are the standard way to create training data for your own corpus.

**Next:** `2.6_parsing_with_llm.ipynb` — LLM parsing for the cases that need comprehension.